In [90]:
from pathlib import Path
import heapq
import io
import struct
import zlib
from collections import Counter

import numpy as np
from PIL import Image

DATASET_NAME = "Bird Speciees Dataset"
ROOT = next(
    (candidate for candidate in (Path(DATASET_NAME), Path("..") / DATASET_NAME)
     if candidate.is_dir()),
    Path(DATASET_NAME),
)


data = []
paths = []
for path in sorted(ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in [".jpg", ".jpeg"]:
        with Image.open(path) as im:
            data.append(np.asarray(im.convert("RGB")))
        paths.append(path)

print(f"Dataset root: {ROOT.resolve()}")
print(f"Loaded {len(data)} images")
if data:
    print("Example:", paths[0], data[0].shape, data[0].dtype)

Dataset root: C:\Users\fredr\Master\høst2026\IDIG4120\Dataset\Digital-information-processing\Bird Speciees Dataset
Loaded 811 images
Example: ..\Bird Speciees Dataset\AMERICAN GOLDFINCH\001.jpg (224, 224, 3) uint8


### Huffman

In [91]:
def huffman_code_lengths(freqs):
    """freqs: dict symbol(int 0..255) -> count. Returns dict symbol -> bit length."""
    symbols = [s for s, f in freqs.items() if f > 0]
    if not symbols:
        return {}
    if len(symbols) == 1:
        return {symbols[0]: 1}   # single-symbol edge case: 1 bit per symbol

    # Deterministic tie-breaking: (freq, insertion order, node)
    heap = [(freqs[s], i, s) for i, s in enumerate(symbols)]
    heapq.heapify(heap)
    tie = len(symbols)

    while len(heap) > 1:
        f1, _, n1 = heapq.heappop(heap)
        f2, _, n2 = heapq.heappop(heap)
        heapq.heappush(heap, (f1 + f2, tie, (n1, n2)))
        tie += 1

    root = heap[0][2]

    lengths = {}
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if isinstance(node, int):
            lengths[node] = depth
        else:
            left, right = node
            stack.append((left, depth + 1))
            stack.append((right, depth + 1))
    return lengths


def canonical_codes(lengths):
    """lengths: dict symbol -> length. Returns dict symbol -> (code_int, length).
    Canonical Huffman: sort by (length, symbol), assign incrementally."""
    items = sorted(lengths.items(), key=lambda kv: (kv[1], kv[0]))
    codes = {}
    code = 0
    prev_len = 0
    for sym, length in items:
        code <<= (length - prev_len)
        codes[sym] = (code, length)
        code += 1
        prev_len = length
    return codes

### Bit-level IO

In [92]:
def encode_symbols(symbols, codes):
    """symbols: iterable of ints. codes: dict sym -> (code, length). MSB-first packing."""
    out = bytearray()
    acc = 0
    nbits = 0
    for s in symbols:
        code, length = codes[s]
        acc = (acc << length) | code
        nbits += length
        while nbits >= 8:
            nbits -= 8
            out.append((acc >> nbits) & 0xFF)
    if nbits > 0:
        out.append((acc << (8 - nbits)) & 0xFF)
    return bytes(out)


def decode_bits(payload, codes, n_symbols):
    """Inverse of encode_symbols. Reads exactly n_symbols symbols."""
    by_len = {}
    for sym, (code, length) in codes.items():
        by_len.setdefault(length, {})[code] = sym

    out = bytearray(n_symbols)
    acc = 0
    nbits = 0
    bit_pos = 0
    total_bits = len(payload) * 8

    for i in range(n_symbols):
        while nbits not in by_len or acc not in by_len[nbits]:
            if bit_pos >= total_bits:
                raise ValueError(f"ran out of bits at symbol {i}/{n_symbols}")
            byte = payload[bit_pos >> 3]
            bit = (byte >> (7 - (bit_pos & 7))) & 1
            acc = (acc << 1) | bit
            nbits += 1
            bit_pos += 1
        out[i] = by_len[nbits][acc]
        acc = 0
        nbits = 0
    return bytes(out)

### Container

In [93]:
MAGIC = b"HFM2"


def pack_header(dtype_str, shape, original_n_bytes, code_lengths, crc, codec_id, predictor=0):
    """Pack the self-describing header. Byte layout, little-endian integers:

    [0:4]        magic "HFM2"
    [4]          dtype string length
    [5:5+len]    dtype string (ascii)
    +1           ndim
    +4*ndim      shape, one uint32 per dimension
    +8           original byte count (uint64)
    +256         Huffman code lengths, one byte per symbol
    +4           CRC32 of the original bytes (uint32)
    +1           codec id: 0 = Huffman, 1 = PNG
    +1           predictor: 0 = none, 1 = horizontal, 2 = vertical
    everything after the header is the payload (Huffman bits or PNG image)

    A 224x224x3 uint8 image has a 5-char dtype and ndim 3, so its header is
    4+1+5+1+12+8+256+4+1+1 = 293 bytes.
    """
    assert len(code_lengths) == 256
    dtype_bytes = dtype_str.encode("ascii")
    out = bytearray()
    out += MAGIC
    out.append(len(dtype_bytes))
    out += dtype_bytes
    out.append(len(shape))
    for d in shape:
        out += struct.pack("<I", d)
    out += struct.pack("<Q", original_n_bytes)
    out += bytes(code_lengths)
    out += struct.pack("<I", crc)
    out.append(codec_id)
    out.append(predictor)
    return bytes(out)


def unpack_header(buf):
    """Inverse of pack_header. Returns (dtype_str, shape, original_n_bytes,
    code_lengths, crc, codec_id, predictor, payload)."""
    if buf[:4] != MAGIC:
        raise ValueError("bad magic or unsupported codec version")
    pos = 4
    dtype_len = buf[pos]
    pos += 1
    dtype_str = buf[pos:pos + dtype_len].decode("ascii")
    pos += dtype_len
    ndim = buf[pos]
    pos += 1
    shape = []
    for _ in range(ndim):
        shape.append(struct.unpack_from("<I", buf, pos)[0])
        pos += 4
    shape = tuple(shape)
    original_n_bytes = struct.unpack_from("<Q", buf, pos)[0]
    pos += 8
    code_lengths = list(buf[pos:pos + 256])
    pos += 256
    crc = struct.unpack_from("<I", buf, pos)[0]
    pos += 4
    codec_id = buf[pos]
    pos += 1
    predictor = buf[pos]
    pos += 1
    return dtype_str, shape, original_n_bytes, code_lengths, crc, codec_id, predictor, buf[pos:]


### Predictors (delta pre-processing, Huffman path only)

In [94]:
# Delta pre-processing for the Huffman path: the transform is applied to the
# samples before byte counting / Huffman coding. The PNG path never uses it.
# uint8 arithmetic wraps mod 256, so encoder and decoder are exact inverses.
PREDICTOR_IDS = {"none": 0, "horizontal": 1, "vertical": 2}
PREDICTOR_NAMES = {v: k for k, v in PREDICTOR_IDS.items()}


def delta_horizontal(arr):
    """Horizontal delta: each sample minus its left neighbour (axis 1).
    The first column is stored verbatim; uint8 subtraction wraps mod 256."""
    out = np.empty_like(arr)
    out[:, 0] = arr[:, 0]
    out[:, 1:] = arr[:, 1:] - arr[:, :-1]
    return out


def delta_vertical(arr):
    """Vertical delta: each sample minus its upper neighbour (axis 0).
    The first row is stored verbatim; uint8 subtraction wraps mod 256."""
    out = np.empty_like(arr)
    out[0, :] = arr[0, :]
    out[1:, :] = arr[1:, :] - arr[:-1, :]
    return out


def undelta_horizontal(arr):
    """Exact inverse of delta_horizontal: cumulative sum along axis 1.
    The uint8 accumulator wraps mod 256, mirroring the subtraction."""
    return np.cumsum(arr, axis=1, dtype=np.uint8)


def undelta_vertical(arr):
    """Exact inverse of delta_vertical: cumulative sum along axis 0."""
    return np.cumsum(arr, axis=0, dtype=np.uint8)


## Interface

In [95]:
def _is_png_image(arr):
    return (arr.dtype == np.uint8 and arr.ndim in (2, 3)
            and (arr.ndim == 2 or arr.shape[2] in (3, 4)))


def _is_predictable(arr):
    """The delta predictors are defined for 2-D or 3-D uint8 arrays."""
    return arr.dtype == np.uint8 and arr.ndim in (2, 3)


def _huffman_container(arr, raw, n, predictor_id):
    """Build one complete Huffman container using a given predictor.

    The predictor transforms the samples first; Huffman then codes the
    transformed bytes. The CRC always covers the *original* bytes so it can
    be checked after decoding and inverting the predictor."""
    if predictor_id == 0:
        stream = raw
    else:
        predicted = delta_horizontal(arr) if predictor_id == 1 else delta_vertical(arr)
        stream = np.ascontiguousarray(predicted).tobytes()
    freqs = Counter(stream)
    lengths = huffman_code_lengths(freqs)
    codes = canonical_codes(lengths)
    payload = encode_symbols(stream, codes)
    code_lengths = [lengths.get(s, 0) for s in range(256)]
    header = pack_header(str(arr.dtype), arr.shape, n, code_lengths,
                         zlib.crc32(raw) & 0xFFFFFFFF, 0, predictor_id)
    return header + payload


def compress(data, settings=None):
    """Return a self-contained, lossless compressed representation of a NumPy array.

    settings:
      "image_codec": "auto" (default) | "huffman" | "png"
      "predictor":   "none" (default) | "horizontal" | "vertical" | "auto"
        Delta pre-processing for the Huffman path only; the PNG path ignores
        it. On non-image arrays horizontal/vertical are ignored (the header
        then records "none"); "auto" tries all applicable variants and keeps
        the smallest container.
    """
    arr = np.ascontiguousarray(data)
    raw = arr.tobytes()
    n = len(raw)
    settings = settings or {}
    image_codec = settings.get("image_codec", "auto")
    predictor = settings.get("predictor", "none")
    if predictor not in ("none", "horizontal", "vertical", "auto"):
        raise ValueError(f"unknown predictor: {predictor!r}")

    # Empty array: only header needed (nothing to predict)
    if n == 0:
        crc = zlib.crc32(raw) & 0xFFFFFFFF
        return pack_header(str(arr.dtype), arr.shape, 0, [0] * 256, crc, 0, 0)

    # Which delta variants the Huffman path should try. Explicit predictors
    # are only defined on image-shaped arrays; elsewhere they are ignored,
    # and "auto" degenerates to "none" as well.
    if _is_predictable(arr) and predictor == "auto":
        candidates = ("none", "horizontal", "vertical")
    elif _is_predictable(arr) and predictor in ("horizontal", "vertical"):
        candidates = (predictor,)
    else:
        candidates = ("none",)

    # --- Compute Huffman results if needed (one per predictor candidate) ---
    huffman_results = None
    if image_codec != "png":          # "huffman" or "auto"
        huffman_results = {
            name: _huffman_container(arr, raw, n, PREDICTOR_IDS[name])
            for name in candidates
        }

    # --- Compute PNG result if needed (predictor never applies here) ---
    png_result = None
    if image_codec != "huffman" and _is_png_image(arr):
        image = Image.fromarray(arr)
        output = io.BytesIO()
        image.save(output, format="PNG", optimize=True, compress_level=9)
        payload = output.getvalue()
        header = pack_header(str(arr.dtype), arr.shape, n, [0] * 256,
                             zlib.crc32(raw) & 0xFFFFFFFF, 1, 0)
        png_result = header + payload

    # --- Select the output ---
    if image_codec == "png":
        if png_result is None:
            raise ValueError("PNG codec requested but array is not a valid PNG image")
        return png_result
    elif image_codec == "huffman":
        return min(huffman_results.values(), key=len)
    else:  # auto
        built = [r for r in (png_result, *huffman_results.values()) if r is not None]
        # If PNG is not applicable, fall back to the best Huffman container
        return min(built, key=len) if built else min(huffman_results.values(), key=len)



def decompress(compressed, settings_or_metadata=None):
    """Return the original array from a losslessly compressed byte string."""
    (dtype_str, shape, original_n_bytes, code_lengths, crc,
     codec_id, predictor, payload) = unpack_header(compressed)

    if predictor not in PREDICTOR_NAMES:
        raise ValueError(f"unknown predictor id: {predictor}")
    if codec_id == 1 and predictor != 0:
        raise ValueError("PNG containers do not use a predictor")

    if original_n_bytes == 0:
        raw = b""
    elif codec_id == 1:
        with Image.open(io.BytesIO(payload)) as image:
            decoded = np.asarray(image)
        if decoded.dtype != np.uint8 or decoded.shape != shape:
            raise ValueError("PNG payload does not match the stored array metadata")
        raw = np.ascontiguousarray(decoded).tobytes()
    elif codec_id == 0:
        lengths = {s: L for s, L in enumerate(code_lengths) if L > 0}
        if not lengths:
            raise ValueError("no code lengths in header")
        stream = decode_bits(payload, canonical_codes(lengths), original_n_bytes)
        # Rebuild the (possibly predicted) array, invert the predictor, and
        # only then check length and CRC against the *original* bytes.
        arr = np.frombuffer(stream, dtype=np.dtype(dtype_str)).reshape(shape)
        if predictor != 0 and not _is_predictable(arr):
            raise ValueError("predictor byte set on a non-image-shaped array")
        if predictor == 1:
            arr = undelta_horizontal(arr)
        elif predictor == 2:
            arr = undelta_vertical(arr)
        raw = np.ascontiguousarray(arr).tobytes()
    else:
        raise ValueError(f"unknown codec id: {codec_id}")

    if len(raw) != original_n_bytes:
        raise ValueError("decoded byte count does not match metadata")
    if (zlib.crc32(raw) & 0xFFFFFFFF) != crc:
        raise ValueError("CRC mismatch - payload corrupted")

    return np.frombuffer(raw, dtype=np.dtype(dtype_str)).reshape(shape).copy()


### Byte accounting helper

In [96]:
def byte_report(arr, compressed):
    orig = arr.nbytes
    comp = len(compressed)
    return {
        "original_bytes": orig,
        "compressed_bytes": comp,
        "ratio": orig / comp if comp else float("nan"),
        "bits_per_input_byte": 8 * comp / orig if orig else float("nan"),
    }


def print_report(label, arr, compressed):
    r = byte_report(arr, compressed)
    print(f"{label:30s} "
          f"orig={r['original_bytes']:>8d}  "
          f"comp={r['compressed_bytes']:>8d}  "
          f"ratio={r['ratio']:.3f}  "
          f"bits/byte={r['bits_per_input_byte']:.3f}")


def describe_compressed(compressed):
    """Name the codec a container actually used, read from its header."""
    *_, codec_id, _predictor, _payload = unpack_header(compressed)
    return {0: "huffman", 1: "png"}.get(codec_id, f"unknown codec id {codec_id}")


def describe_predictor(compressed):
    """Name the delta predictor a container actually used, read from its header."""
    *_, predictor, _payload = unpack_header(compressed)
    return PREDICTOR_NAMES.get(predictor, f"unknown predictor id {predictor}")


def container_sizes(compressed):
    """Split one of our containers into header (side info) and payload."""
    *_, payload = unpack_header(compressed)
    return len(compressed) - len(payload), len(payload)


###  Round-trip tests (shape, dtype, empty, tiny, unseen)

In [97]:
def roundtrip(arr, label="", settings=None):
    comp = compress(arr, settings)
    dec = decompress(comp)
    assert dec.shape == arr.shape, f"shape mismatch {label}: {dec.shape} vs {arr.shape}"
    assert dec.dtype == arr.dtype, f"dtype mismatch {label}: {dec.dtype} vs {arr.dtype}"
    assert np.array_equal(dec, arr), f"data mismatch {label}"
    return comp

rng = np.random.default_rng(0)

tests = [
    ("empty uint8",       np.zeros((0,), dtype=np.uint8)),
    ("tiny 1 elem",       np.array([42], dtype=np.uint8)),
    ("tiny 1x1 float32",  np.array([[3.14]], dtype=np.float32)),
    ("1-D int16",         np.arange(16, dtype=np.int16)),
    ("2D uint8 ramp",     np.arange(256, dtype=np.uint8).reshape(16, 16)),
    ("3D uint8 uniform",  rng.integers(0, 256, size=(8, 8, 3), dtype=np.uint8)),
    ("float32 noise",     rng.standard_normal((32, 32)).astype(np.float32)),
    ("all same symbol",   np.full((100,), 7, dtype=np.uint8)),
]

for label, arr in tests:
    comp = roundtrip(arr, label)
    print_report(label, arr, comp)

image = np.zeros((256, 256, 3), dtype=np.uint8)
image[:, :, 0] = np.arange(256, dtype=np.uint8)[:, None]
image[:, :, 1] = np.arange(256, dtype=np.uint8)[None, :]
image[:, :, 2] = 127
image_comp = roundtrip(image, "synthetic RGB image")
print_report(f"synthetic RGB image [{describe_compressed(image_comp)}]",
            image, image_comp)

corrupt = bytearray(image_comp)
corrupt[len(corrupt) // 2] ^= 1
try:
    decompress(bytes(corrupt))
except (ValueError, OSError):
    print("png corruption check: passed (Pillow rejected the payload)")
else:
    raise AssertionError("corrupted png payload was accepted")

huff_comp = compress(image, {"image_codec": "huffman"})
corrupt_h = bytearray(huff_comp)
corrupt_h[len(corrupt_h) // 2] ^= 1
try:
    decompress(bytes(corrupt_h))
except ValueError:
    print("huffman corruption check: passed (CRC mismatch)")
else:
    raise AssertionError("corrupted huffman payload was accepted")

empty uint8                    orig=       0  comp=     285  ratio=0.000  bits/byte=nan
tiny 1 elem                    orig=       1  comp=     286  ratio=0.003  bits/byte=2288.000
tiny 1x1 float32               orig=       4  comp=     292  ratio=0.014  bits/byte=584.000
1-D int16                      orig=      32  comp=     297  ratio=0.108  bits/byte=74.250
2D uint8 ramp                  orig=     256  comp=     369  ratio=0.694  bits/byte=11.531
3D uint8 uniform               orig=     192  comp=     461  ratio=0.416  bits/byte=19.208
float32 noise                  orig=    4096  comp=    4043  ratio=1.013  bits/byte=7.896
all same symbol                orig=     100  comp=     298  ratio=0.336  bits/byte=23.840
synthetic RGB image [png]      orig=  196608  comp=     864  ratio=227.556  bits/byte=0.035
png corruption check: passed (Pillow rejected the payload)
huffman corruption check: passed (CRC mismatch)


### Predictor tests (Huffman path only)

In [98]:
# 1) The delta transforms and their inverses are exact inverses on real images.
for sample in (data[:2] if data else [image]):
    assert np.array_equal(undelta_horizontal(delta_horizontal(sample)), sample)
    assert np.array_equal(undelta_vertical(delta_vertical(sample)), sample)
print("predictor inverse check: passed (exact on real images)")

# 2) Round-trips with an explicit predictor on image-shaped arrays.
for pred in ("horizontal", "vertical"):
    comp = roundtrip(image, f"synthetic RGB image, predictor {pred}",
                     {"image_codec": "huffman", "predictor": pred})
    assert describe_predictor(comp) == pred
gray = np.arange(256, dtype=np.uint8).reshape(16, 16)  # 2-D image-shaped ramp
for pred in ("horizontal", "vertical"):
    comp = roundtrip(gray, f"2D uint8 ramp, predictor {pred}",
                     {"image_codec": "huffman", "predictor": pred})
    assert describe_predictor(comp) == pred

# 3) Non-image arrays: the predictor is not defined there, so it is ignored
#    and the header records "none" (verified on decode).
for label, arr, pred in [
    ("1-D int16", np.arange(16, dtype=np.int16), "horizontal"),
    ("1-D uint8", np.full((100,), 7, dtype=np.uint8), "vertical"),
    ("2-D float32", np.linspace(0.0, 1.0, 64, dtype=np.float32).reshape(8, 8),
     "horizontal"),
]:
    comp = roundtrip(arr, f"{label}, predictor {pred} (non-image)",
                     {"predictor": pred})
    assert describe_predictor(comp) == "none"
print("non-image predictor check: passed (ignored, header records none)")

# 4) Unknown predictor names and header ids are rejected.
try:
    compress(image, {"predictor": "diagonal"})
except ValueError:
    print("invalid predictor name rejected: passed")
else:
    raise AssertionError("invalid predictor name accepted")
header_len, _ = container_sizes(huff_comp)
bad = bytearray(huff_comp)
bad[header_len - 1] = 3  # the predictor byte is the last byte of the header
try:
    decompress(bytes(bad))
except ValueError:
    print("invalid predictor id rejected: passed")
else:
    raise AssertionError("invalid predictor id accepted")

# 5) The corruption check also catches damage in a predicted container.
pred_comp = compress(image, {"image_codec": "huffman", "predictor": "horizontal"})
corrupt_p = bytearray(pred_comp)
corrupt_p[len(corrupt_p) // 2] ^= 1
try:
    decompress(bytes(corrupt_p))
except ValueError:
    print("huffman+predictor corruption check: passed (CRC mismatch)")
else:
    raise AssertionError("corrupted huffman+predictor payload was accepted")


predictor inverse check: passed (exact on real images)
non-image predictor check: passed (ignored, header records none)
invalid predictor name rejected: passed
invalid predictor id rejected: passed
huffman+predictor corruption check: passed (CRC mismatch)


### Predictor comparison on the development image

In [99]:
# The predictor only ever affects the Huffman path, so force that codec to
# measure what the delta pre-processing buys on a real image.
if data:
    dev_img, dev_name = data[0], paths[0].name
else:  # synthetic fallback keeps the cell meaningful without the dataset
    dev_img, dev_name = image, "synthetic image"

pred_containers = {}
for pred in ("none", "horizontal", "vertical"):
    comp = roundtrip(dev_img, f"{dev_name} predictor {pred}",
                     {"image_codec": "huffman", "predictor": pred})
    assert describe_predictor(comp) == pred
    pred_containers[pred] = comp
    print_report(f"huffman + {pred}", dev_img, comp)

# Auto mode must return exactly the smallest of the three, and its header
# must say which one that is (read back through describe_predictor).
c_auto = roundtrip(dev_img, f"{dev_name} predictor auto",
                   {"image_codec": "huffman", "predictor": "auto"})
best = min(pred_containers, key=lambda p: len(pred_containers[p]))
assert len(c_auto) == len(pred_containers[best])
assert describe_predictor(c_auto) == best
sizes = {p: len(c) for p, c in pred_containers.items()}
print(f"predictor auto picked {best!r}: {len(c_auto)} bytes, the smallest of {sizes}")

# Full auto (codec + predictor together) is never larger than predictor "none".
c_full_auto = roundtrip(dev_img, f"{dev_name} full auto", {"predictor": "auto"})
assert len(c_full_auto) <= len(compress(dev_img, {"predictor": "none"}))
print_report(f"auto codec+predictor [{describe_compressed(c_full_auto)}+"
             f"{describe_predictor(c_full_auto)}]", dev_img, c_full_auto)


huffman + none                 orig=  150528  comp=  144569  ratio=1.041  bits/byte=7.683
huffman + horizontal           orig=  150528  comp=   94637  ratio=1.591  bits/byte=5.030
huffman + vertical             orig=  150528  comp=   99454  ratio=1.514  bits/byte=5.286
predictor auto picked 'horizontal': 94637 bytes, the smallest of {'none': 144569, 'horizontal': 94637, 'vertical': 99454}
auto codec+predictor [png+none] orig=  150528  comp=   81979  ratio=1.836  bits/byte=4.357


### Round-trip on dataset images

In [100]:
if data:
    # Develop on the first image. Auto mode picks the container codec;
    # the label discloses which one was actually used.
    train_img = data[0]
    comp = roundtrip(train_img, paths[0].name)
    print_report(f"{paths[0].name} [{describe_compressed(comp)}]",
                 train_img, comp)

    # The same image through the forced Huffman path, so the Huffman
    # numbers on real data are measured, not implied by auto mode.
    comp_h = roundtrip(train_img, paths[0].name, {"image_codec": "huffman"})
    print_report(f"{paths[0].name} [huffman]", train_img, comp_h)

    # Unseen file: last image, not used during development. Both the
    # auto mode and the forced Huffman path are reported for it.
    if len(data) > 1:
        unseen = data[-1]
        comp_u = roundtrip(unseen, paths[-1].name + " (unseen)")
        print_report(f"{paths[-1].name} (unseen) [{describe_compressed(comp_u)}]",
                     unseen, comp_u)
        comp_uh = roundtrip(unseen, paths[-1].name + " (unseen)",
                            {"image_codec": "huffman"})
        print_report(f"{paths[-1].name} (unseen) [huffman]", unseen, comp_uh)

    compressed_data = comp
    print(f"compressed_data type: {type(compressed_data).__name__}")
    print(f"compressed_data length: {len(compressed_data)} bytes")
    print(f"first 64 bytes (hex): {compressed_data[:64].hex()}")

001.jpg [png]                  orig=  150528  comp=   81979  ratio=1.836  bits/byte=4.357
001.jpg [huffman]              orig=  150528  comp=  144569  ratio=1.041  bits/byte=7.683
132.jpg (unseen) [png]         orig=  150528  comp=   84397  ratio=1.784  bits/byte=4.485
132.jpg (unseen) [huffman]     orig=  150528  comp=  144854  ratio=1.039  bits/byte=7.698
compressed_data type: bytes
compressed_data length: 81979 bytes
first 64 bytes (hex): 48464d320575696e743803e0000000e000000003000000004c020000000000000000000000000000000000000000000000000000000000000000000000000000


## Baseline comparison


In [101]:
def baseline_zlib(arr, level=6):
    raw = np.ascontiguousarray(arr).tobytes()
    return zlib.compress(raw, level=level)


if data:
    img = data[0]
    n_raw = img.nbytes

    # Ours in forced Huffman mode: byte-level canonical Huffman coding of
    # the raw byte stream, i.e. exactly the same input the baseline sees.
    ours_huffman = compress(img, {"image_codec": "huffman"})
    assert describe_compressed(ours_huffman) == "huffman"

    # Same coder, but with a horizontal delta applied to the samples before
    # byte counting. The Huffman code itself is unchanged; only the input
    # distribution is different.
    ours_huff_h = compress(img, {"image_codec": "huffman", "predictor": "horizontal"})
    assert describe_compressed(ours_huff_h) == "huffman"
    assert describe_predictor(ours_huff_h) == "horizontal"

    # Ours in PNG mode: a different design (2-D filters + DEFLATE inside
    # the PNG). Reported separately so it is never confused with the
    # Huffman numbers. This is what auto mode picks for images.
    ours_png = compress(img, {"image_codec": "png"})
    assert describe_compressed(ours_png) == "png"

    # Baseline: zlib at its default level 6, on the exact same raw byte
    # stream as our no-predictor Huffman.
    base = baseline_zlib(img, level=6)

    # Full auto (codec + predictor together): the smallest container our
    # codec can produce for this image, and what it actually chooses.
    ours_full_auto = compress(img, {"predictor": "auto"})

    print("Comparison on:", paths[0].name,
          f"({img.dtype}, shape {img.shape}, {n_raw} raw bytes)")
    print("Every size below is a complete output: our totals include our")
    print("container header, and the zlib total includes zlib's own ~6-byte")
    print("framing. No side information is counted separately.\n")

    print_report("huffman (raw bytes)", img, ours_huffman)
    print_report("huffman + horizontal", img, ours_huff_h)
    print_report("png", img, ours_png)
    print_report("baseline zlib-6", img, base)
    print_report(f"auto [{describe_compressed(ours_full_auto)}+"
                 f"{describe_predictor(ours_full_auto)}]", img, ours_full_auto)

    header_bytes, payload_bytes = container_sizes(ours_huffman)
    print(f"\nHuffman accounting: header (side info) = {header_bytes} bytes, "
          f"payload = {payload_bytes} bytes.")

    # Zero-order byte entropy of the ORIGINAL bytes: the bound for the
    # no-predictor Huffman path.
    freqs = Counter(np.ascontiguousarray(img).tobytes())
    total = sum(freqs.values())
    entropy = -sum((c / total) * np.log2(c / total) for c in freqs.values())
    print(f"Zero-order byte entropy (raw bytes): {entropy:.3f} bits/byte "
          f"(ideal ~{int(entropy * n_raw / 8)} bytes). Our Huffman payload is "
          f"{8 * payload_bytes / n_raw:.3f} bits/byte, i.e. close to its bound.")

    # Same calculation for the horizontally predicted stream, so the
    # improvement from the predictor is measured against its own bound.
    pred_stream = np.ascontiguousarray(delta_horizontal(img)).tobytes()
    pf = Counter(pred_stream)
    pt = sum(pf.values())
    pred_entropy = -sum((c / pt) * np.log2(c / pt) for c in pf.values())
    pred_payload = len(ours_huff_h) - header_bytes
    print(f"Zero-order byte entropy (horizontal delta): {pred_entropy:.3f} bits/byte "
          f"(ideal ~{int(pred_entropy * n_raw / 8)} bytes). "
          f"Our Huffman+horizontal payload is "
          f"{8 * pred_payload / n_raw:.3f} bits/byte.")

    print("Fair reading: our Huffman and zlib-6 compressed the same raw byte")
    print("stream. zlib is smaller because LZ77 matching also removes")
    print("repetition, which a zero-order byte model cannot represent. The")
    print("horizontal-delta row uses the same Huffman coder on a better input")
    print("distribution, which isolates the model gap from the coder.")

Comparison on: 001.jpg (uint8, shape (224, 224, 3), 150528 raw bytes)
Every size below is a complete output: our totals include our
container header, and the zlib total includes zlib's own ~6-byte
framing. No side information is counted separately.

huffman (raw bytes)            orig=  150528  comp=  144569  ratio=1.041  bits/byte=7.683
huffman + horizontal           orig=  150528  comp=   94637  ratio=1.591  bits/byte=5.030
png                            orig=  150528  comp=   81979  ratio=1.836  bits/byte=4.357
baseline zlib-6                orig=  150528  comp=  105908  ratio=1.421  bits/byte=5.629
auto [png+none]                orig=  150528  comp=   81979  ratio=1.836  bits/byte=4.357

Huffman accounting: header (side info) = 293 bytes, payload = 144276 bytes.
Zero-order byte entropy (raw bytes): 7.648 bits/byte (ideal ~143906 bytes). Our Huffman payload is 7.668 bits/byte, i.e. close to its bound.
Zero-order byte entropy (horizontal delta): 4.984 bits/byte (ideal ~93782 bytes). 

### Failure case

In [102]:
# 1) Uniform random bytes: entropy is maximal, Huffman cannot shrink it.
random_bytes = rng.integers(0, 256, size=100_000, dtype=np.uint8)
c = compress(random_bytes)
print_report("uniform random 100k", random_bytes, c)
# Expect: ratio slightly < 1 because of the ~290-byte header.

# 2) Already-compressed data (JPEG file on disk): same situation.
if paths:
    jpeg_bytes = np.frombuffer(paths[0].read_bytes(), dtype=np.uint8)
    c2 = compress(jpeg_bytes)
    print_report("raw JPEG bytes", jpeg_bytes, c2)
    # Expect: ratio <= 1, i.e. no useful compression.

# 3) Tiny inputs: header dominates.
tiny = np.array([1, 2, 3], dtype=np.uint8)
print_report("3-byte input", tiny, compress(tiny))
# Expect: ratio << 1 (expansion). Document this as "do not use Huffman on tiny inputs".

uniform random 100k            orig=  100000  comp=  100285  ratio=0.997  bits/byte=8.023
raw JPEG bytes                 orig=   22236  comp=   22456  ratio=0.990  bits/byte=8.079
3-byte input                   orig=       3  comp=     286  ratio=0.010  bits/byte=762.667
